In [8]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.special import softmax
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

In [9]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [10]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Available Devices :  [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]


# Chapter 13: Best Practices for the Real World

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

If you are to go out in the real world and achieve state-of-the-art results
on brand new problems, there’s still a bit of a chasm that you’ll need to cross.
We’ll review essential techniques for systematically improving
model performance: hyperparameter tuning and model ensembling. Then we’ll
look at how you can speed up and scale up model training, with multi-GPU and TPU
training, mixed precision, and leveraging remote computing resources in the cloud.

## 13.1 Getting the Most out of your Models

### 13.1.1 Hyperparameter Optimization

Aarchitecture-level parameters are called hyperparameters to distinguish them
from the parameters of a model, which are trained via backpropagation.

You need to explore the space of possible decisions automatically, systematically,
in a principled way. You need to search the architecture space and find the bestperforming
architectures empirically. That’s what the field of automatic hyperparameter
optimization is about: it’s an entire field of research, and an important one.
The process of optimizing hyperparameters typically looks like this:

1. Choose a set of hyperparameters (automatically).
2. Build the corresponding model.
3. Fit it to your training data, and measure performance on the validation data.
4. Choose the next set of hyperparameters to try (automatically).
5. Repeat.
6. Eventually, measure performance on your test data.

Consider these points:

- The hyperparameter space is typically made up of discrete decisions and thus
  isn’t continuous or differentiable. Hence, you typically can’t do gradient descent
  in hyperparameter space. Instead, you must rely on gradient-free optimization
  techniques, which naturally are far less efficient than gradient descent.
- Computing the feedback signal of this optimization process (does this set of
  hyperparameters lead to a high-performing model on this task?) can be extremely
  expensive: it requires creating and training a new model from scratch on your
  dataset.
- The feedback signal may be noisy: if a training run performs 0.2% better, is that
  because of a better model configuration, or because you got lucky with the initial
  weight values?


#### Using `KerasTuner`

To specify a search space, define a model-building function.
It takes an hp argument, from which you can sample hyperparameter ranges, and it
returns a compiled Keras model.

In [1]:
def build_model(hp):
    """Sample hyperparameter values from the hp (hyper parameters) object,
    After sampling, these values (such as number of hidden units for a layer here)
    are just regular Python constants.
    """
    units = hp.Int(name="units", min_value=16, max_value=64, step=16)
    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    # different kinds of hyperparameters are available besides Int, like Float
    # Boolean, Choice
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])

    # the function returns a compiled model
    return model

If you want to adopt a more modular and configurable approach to model-building,
you can also subclass the HyperModel class and define a build method, as follows.

In [5]:
import keras_tuner as kt

class SimpleMLP(kt.HyperModel):
    def __init__(self, num_classes):
        self.num_classes = num_classes
        
    def build(self, hp):
        """The build method is identical to the previous regular standalone function
        """
        units = hp.Int(name="units", min_value=16, max_value=64, step=16)
        # we can configure model constraints as constructo arguments, here self.num_classes
        # can vary in the models we want to tune
        model = keras.Sequential([
            layers.Dense(units, activation="relu"),
            layers.Dense(self.num_classes, activation="softmax")
        ])
        
        optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
        
        return model


In [6]:
hypermodel = SimpleMLP(num_classes=10)

The next step is to define a “tuner.” Schematically, you can think of a tuner as a for
loop that will repeatedly

- Pick a set of hyperparameter values
- Call the model-building function with these values to create a model
- Train the model and record its metrics

KerasTuner has several built-in tuners available—RandomSearch, BayesianOptimization,
and Hyperband. Let’s try BayesianOptimization, a tuner that attempts to make
smart predictions for which new hyperparameter values are likely to perform best
given the outcomes of previous choices.

In [11]:
tuner = kt.BayesianOptimization(
    # specify the model-building function or could use HyperModel instance
    build_model,
    # Specify the metric that the tuner will seek to optimize.  Always specify validation metrics,
    # since the goal of the search process is to find models that generalize
    objective="val_accuracy",
    # maximum number of different model configurations ('trials') to try before ending the
    # search
    max_trials=100,
    # to reduce metrics variance, you can train the same model multiple times and average the
    # results.  executions_per_trial is how many training rounds (executions) to run for each
    # model configuration (trial)
    executions_per_trial=2,
    # where to store search logs
    directory="../models/mnist_kt_test",
    # whether to overwrite data in directory to start a new search.  Set this to True if you've
    # modified the model-building function, or to False to resume a previously started search
    # with the same model building function.
    overwrite=True,
)

You can display an overview of the search space via `search_space_summary()`

In [12]:
tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 16, 'sampling': 'linear'}
optimizer (Choice)
{'default': 'rmsprop', 'conditions': [], 'values': ['rmsprop', 'adam'], 'ordered': False}


Let's launch the search.  Don't forget to pass validation data and make sure not
to use your test set s validation data.

In [13]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255

# make a copy of the full training data for later
x_train_full = x_train[:]
y_train_full = y_train[:]

# split 10000 samples to use for validation
num_val_samples = 10000
x_train, x_val = x_train[:-num_val_samples], x_train[-num_val_samples:]
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:]

In [14]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
]

# notice search is similar to fit, this defines a loop that creates, compiles and
# fits models using different hyperparamter values
# this function takes the same arguments as fit(), it simply passes them down
# to fit() for each new model
tuner.search(
    x_train, y_train,
    batch_size=128,
    # Use a large number of epochs (you don't know in advance how many epochs each model will need), and use an
    # EarlyStopping callback to stop training when you start overfitting
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2,
)

Trial 100 Complete [00h 00m 53s]
val_accuracy: 0.9745000004768372

Best val_accuracy So Far: 0.9770999848842621
Total elapsed time: 01h 35m 05s


In [15]:
tuner.results_summary()

Results summary
Results in ../models/mnist_kt_test/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 066 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9770999848842621

Trial 013 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9765999913215637

Trial 088 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.976500004529953

Trial 036 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9764499962329865

Trial 047 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9763500094413757

Trial 092 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9763500094413757

Trial 087 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9760999977588654

Trial 052 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9758999943733215

Trial 078 summary
Hyperparameters:
units: 64
optimizer: rmsprop
Score: 0.9758999943733215

Trial 038 summary
Hyperparameters:
units: 

The preceeding had 4 different settings for the number of units, and two different choices for the
optimizer, so there should only be 8 models that will be trained.  But actually the keras tuner
seems to do the specified `max_trials`.

If your search process crashes, you can always
restart it—just specify overwrite=False in the tuner so that it can resume from the
trial logs stored on disk.

Once the search is complete, you can query the best hyperparameter configurations,
which you can use to create high-performing models that you can then retrain.


In [16]:
top_n = 4

# returns a list of HyperParameter objects, which you can pass to the model-building
# function
best_hps = tuner.get_best_hyperparameters(top_n)

In [19]:
best_hps[0].values
best_hps[0].

{'units': 64, 'optimizer': 'rmsprop'}

Usually, when retraining these models, you may want to include the validation data as
part of the training data, since you won’t be making any further hyperparameter
changes, and thus you will no longer be evaluating performance on the validation
data. In our example, we’d train these final models on the totality of the original
MNIST training data, without reserving a validation set.

There’s one last parameter we need to settle: the optimal number of epochs to train for. Typically, you’ll want to train
the new models for longer than you did during the search: using an aggressive
patience value in the EarlyStopping callback saves time during the search, but it may
lead to under-fit models. Just use the validation set to find the best epoch:

In [24]:
def get_best_epoch(hp):
    model = build_model(hp)
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
    ]

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=100,
        batch_size=128,
        callbacks=callbacks,
        verbose=False)
    
    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    
    return best_epoch    

Finally, train on the full dataset for just a bit longer than this epoch count, since
you’re training on more data; 20% more in this case.

In [25]:
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(
        x_train_full, y_train_full,
        batch_size=128, epochs=int(best_epoch * 1.2))
    return model
    
best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(x_test, y_test)
    best_models.append(model)

Best epoch: 13
Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8341 - loss: 0.6422
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9360 - loss: 0.2244
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9543 - loss: 0.1587
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9642 - loss: 0.1273
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9700 - loss: 0.1040
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9744 - loss: 0.0892
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9755 - loss: 0.0816
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9806 - loss: 0.0663
Epoch 9/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9817 - loss: 0.0619
Epoch 10/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9838 - loss: 0.0567
Epoch 11/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9867 - loss: 0.0477
Epoch 12/15
469/469 ━━━━━━━━━━━━━━━━━

Note that if you’re not worried about slightly underperforming, there’s a shortcut you
can take: just use the tuner to reload the top-performing models with the best weights
saved during the hyperparameter search, without retraining new models from scratch.

In [26]:
best_models = tuner.get_best_models(top_n)

/opt/conda/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


#### The art of crafting the right search space

Deep learning automates the task of hierarchical feature engineering—features
are learned using a feedback signal, not hand-tuned, and that’s the way it should be.
In the same way, you shouldn’t handcraft your model architectures; you should optimize
them in a principled way.

However doing hyperparameter tuning is not a replacement for being familiar
with model architecture best practices. Search spaces grow combinatorially with
the number of choices.

You still need to handpick experiment configurations that have the potential to yield good
metrics.

KerasTuner attempts to provide premade search spaces that are
relevant to broad categories of problems, such as image classification. Just add data,
run the search, and get a pretty good model. You can try the hypermodels kt.applications.
HyperXception and kt.applications.HyperResNet, which are effectively
tunable versions of Keras Applications models.

#### The future of hyperparameter tuning: Automated machine learning

Always look at the big picture, focus on understanding the fundamentals, and keep
in mind that the highly specialized tedium will eventually be automated away.

### 13.1.2 Model Ensembling

Another powerful technique for obtaining the best possible results on a task is 
**model ensembling**. Ensembling consists of pooling together the predictions of a set of different
models to produce better predictions.

Ensembling relies on the assumption that different well-performing models
trained independently are likely to be good for different reasons: each model looks at
slightly different aspects of the data to make its predictions, getting part of the “truth”
but not all of it.

The easiest way to pool the predictions of a set
of classifiers (to ensemble the classifiers) is to average their predictions at inference time:

```python
preds_a = model_a.predict(x_val)
preds_b = model_b.predict(x_val)
preds_c = model_c.predict(x_val)
preds_d = model_d.predict(x_val)
final_preds = 0.25 * (preds_a + preds_b + preds_c + preds_d)
```

However, this will work only if the classifiers are more or less equally good. If one of
them is significantly worse than the others, the final predictions may not be as good as
the best classifier of the group.

A smarter way to ensemble classifiers is to do a weighted average, where the
weights are learned on the validation data—typically, the better classifiers are given a
higher weight, and the worse classifiers are given a lower weight.

To search for a good set of ensembling weights, you can use random search or a simple optimization algorithm,
such as the Nelder-Mead algorithm:

```python
preds_a = model_a.predict(x_val)
preds_b = model_b.predict(x_val)
preds_c = model_c.predict(x_val)
preds_d = model_d.predict(x_val)
# these weights, (0.5, 0.25, 0.1 and 0.15) are assumed to be learned empirically
final_preds = 0.5 * preds_a + 0.25 * preds_b + 0.1 * preds_c + 0.15 * preds_d
```

The key to making ensembling work is the diversity of the set of classifiers.

You should ensemble models that are as good as possible while being
as different as possible. This typically means using very different architectures or even
different brands of machine learning approaches.



## 13.2 Scaling-up Model Training

In this section, you’ll learn about three ways you can train your models faster:

- Mixed-precision training, which you can use even with a single GPU
- Training on multiple GPUs
- Training on TPUs


### 13.2.1 Speeding Up Training on GPU with Mixed Precision

What if I told you there’s a simple technique you can use to speed up the training of
almost any model by up to 3X, basically for free? It seems too good to but true, and
yet, such a trick does exist. That’s **mixed-precision** training.

#### Understanding floating-point precision

But you could not do the same with float16 weights and computation; the gradient descent process wouldn’t run
smoothly, since you couldn’t represent small gradient updates of around 1e-5 or 1e-6.

You can, however, use a hybrid approach: that’s what mixed precision is about. The
idea is to leverage 16-bit computations in places where precision isn’t an issue, and to
work with 32-bit values in other places to maintain numerical stability. 

Modern GPUs and TPUs feature specialized hardware that can run 16-bit operations much faster and
use less memory than equivalent 32-bits operations. By using these lower-precision
operations whenever possible, you can speed up training on those devices by a significant
factor.

#### Mixed-precision training in practice

When training on a GPU, you can turn on mixed precision like this:

```python
from tensorflow import keras
keras.mixed_precision.set_global_policy("mixed_float16")
```

### 13.2.2 Multi-GPU Training

Training on a single GPU puts a hard bound on how fast you can move. The solution? You could simply
add more GPUs and start doing multi-GPU distributed training.

There are two ways to distribute computation across multiple devices: data parallelism
and model parallelism.

1. **data parallelism** a single model is replicated on multiple devices or multiple
   machines. Each of the model replicas processes different batches of data, and then
   they merge their results.
2. **model parallelism** with model parallelism, different parts of a single model run on different devices,
   processing a single batch of data together at the same time. This works best with models
   that have a naturally parallel architecture, such as models that feature multiple branches.

#### Getting your hands on two or more GPUs

#### Single-host, multi-device synchronous training

Once you’re able to import tensorflow on a machine with multiple GPUs, you’re seconds
away from training a distributed model. It works like this:


```python
# create a distribution strategy object. MirroredStrategy should be your go-to solution
strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

# everything that creates variables should be under the strategy scope. in general
# this is only model construction and calling compile()
with strategy.scope():
    model = get_compiled_model()

# train the model on all available devices
model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    callbacks=callbacks)
```

These few lines implement the most common training setup: single-host, multi-device
synchronous training, also known in TensorFlow as the “mirrored distribution strategy.”

“Single host” means that the different GPUs considered are all on a single machine
(as opposed to a cluster of many machines, each with its own GPU, communicating
over a network). 

“Synchronous training” means that the state of the per-GPU model
replicas stays the same at all times—there are variants of distributed training where
this isn’t the case.

### 13.2.3 TPU Training

Simple example:

```python
from tensorflow import keras
from tensorflow.keras import layers

strategy = tf.distribute.TPUStrategy(tpu)
print(f"Number of replicas: {strategy.num_replicas_in_sync}")

def build_model(input_size):
    inputs = keras.Input((input_size, input_size, 3))
    x = keras.applications.resnet.preprocess_input(inputs)
    x = keras.applications.resnet.ResNet50(weights=None, include_top=False, pooling="max")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

with strategy.scope():
    model = build_model(input_size=32)
```


In our case, let’s train from NumPy arrays in memory—the CIFAR10 dataset:

```python
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
model.fit(x_train, y_train, batch_size=1024)
```

#### Leveraging step fusing to improve TPU utilization



## Summary

<font color='blue'>
    
- You can leverage hyperparameter tuning and KerasTuner to automate the
  tedium out of finding the best model configuration. But be mindful of validationset overfitting!
- An ensemble of diverse models can often significantly improve the quality of
  your predictions.
- You can speed up model training on GPU by turning on mixed precision—
  you’ll generally get a nice speed boost at virtually no cost.
- To further scale your workflows, you can use the tf.distribute.Mirrored-
  Strategy API to train models on multiple GPUs.
- You can even train on Google’s TPUs (available on Colab) by using the TPUStrategy
  API. If your model is small, make sure to leverage step fusing (via
  the compile(…, steps_per_execution=N) argument) in order to fully utilize
  the TPU cores.